# Baseline MBPP Evaluation — HF models (same-size range first)

Evaluates **any Hugging Face causal LM** on MBPP with the same protocol as the PocketCoder
stage evals: MBPP test split · greedy · seed 42 · new-tokens-only decode · syntax = `ast.parse`
(empty = invalid) · full test_list in a silenced hard-kill sandbox · pass@1 + avg test pass +
syntax validity, all with 95% Wilson CIs · JSON with protocol block.

**Prompt = completion-style** (task + one assert as comments, then the reference `def`
signature) — these are base/completion models that never saw an instruction template.

Preset to **GPT-2 small (124M)** — the same-size general baseline. Run once per model by
changing `MODEL_ID`:

| Baseline | MODEL_ID | Size |
|---|---|---|
| GPT-2 small | `gpt2`  ← current | 124M |
| codeparrot-small | `codeparrot/codeparrot-small` | 110M |
| SmolLM2-135M | `HuggingFaceTB/SmolLM2-135M` | 135M |
| Pythia-160M | `EleutherAI/pythia-160m` | 160M |
| GPT-2 XL | `openai-community/gpt2-xl` | 1.5B |
| TinyLlama (base) | `TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T` | 1.1B |


In [6]:
!pip install -q -U datasets transformers accelerate tqdm

In [8]:
# ======================= CONFIG =======================
MODEL_ID       = "openai-community/gpt2-xl"   # <-- change per baseline (see table above)
NUM_PROBLEMS   = None           # None = full test split; 20 = smoke test
MAX_NEW_TOKENS = 256
SEED           = 42
TIMEOUT_S      = 5
OUT_FILE       = f"mbpp_baseline_{MODEL_ID.split('/')[-1]}.json"
# ======================================================
print("Evaluating baseline:", MODEL_ID, "->", OUT_FILE)

Evaluating baseline: openai-community/gpt2-xl -> mbpp_baseline_gpt2-xl.json


## Load model + tokenizer (HF native)

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token   # GPT-2 / codeparrot have no pad token

dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
ctx_len = getattr(model.config, "max_position_embeddings", None) or getattr(model.config, "n_positions", 1024)
print(f"Loaded {MODEL_ID} — {n_params/1e6:.1f}M params, context {ctx_len}, dtype {dtype}, on {device}")

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded openai-community/gpt2-xl — 1557.6M params, context 1024, dtype torch.float16, on cuda


## MBPP test split + completion-style prompt

In [10]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "full", split="test")
if NUM_PROBLEMS:
    mbpp = mbpp.select(range(min(NUM_PROBLEMS, len(mbpp))))
print(f"MBPP test problems: {len(mbpp)}")

def solution_head(ref_code):
    """Reference solution's leading lines up to and incl. the first `def` line."""
    out = []
    for ln in ref_code.split("\n"):
        out.append(ln)
        if ln.lstrip().startswith("def "):
            return "\n".join(out) + "\n"
    return None

def build_prompt(ex):
    head = solution_head(ex["code"])
    if head is None:
        return None, None
    first_test = ex["test_list"][0] if ex["test_list"] else ""
    prompt = f"# {ex['text']}\n# {first_test}\n{head}"
    return prompt, head

p, h = build_prompt(mbpp[0])
print("--- example prompt ---"); print(p)

README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

full/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 87.2kB            

full/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

full/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  116kB            

full/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/374 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/10 [00:00<?, ? examples/s]

MBPP test problems: 500
--- example prompt ---
# Write a python function to remove first and last occurrence of a given character from the string.
# assert remove_Occ("hello","l") == "heo"
def remove_Occ(s,ch): 



## Sandboxed test execution (child process, silenced, hard kill)

In [11]:
import multiprocessing as mp

def _worker(code_s, setup, tests, q):
    import os as _os, sys as _sys
    _dn = open(_os.devnull, "w"); _sys.stdout = _dn; _sys.stderr = _dn
    ns = {}
    try:
        if setup: exec(setup, ns)
        exec(code_s, ns)
        passed = 0
        for t in tests:
            try: exec(t, ns); passed += 1
            except Exception: pass
        q.put((passed, len(tests)))
    except Exception:
        q.put((0, len(tests)))

def run_tests(code_s, setup, tests, timeout_s=TIMEOUT_S):
    ctx = mp.get_context("fork"); q = ctx.Queue()
    p = ctx.Process(target=_worker, args=(code_s, setup, tests, q), daemon=True)
    p.start(); p.join(timeout_s)
    if p.is_alive():
        p.terminate(); p.join(1)
        if p.is_alive(): p.kill(); p.join()
        return 0, len(tests)
    try: return q.get_nowait()
    except Exception: return 0, len(tests)

## Generate + evaluate

Greedy via `model.generate(do_sample=False)`. The completion is cut at the first line that
leaves the function (a non-indented, non-`def` line) so trailing unrelated text doesn't break
`ast.parse` — same effect as PocketCoder's fence cut, adapted to fence-less completion models.

In [12]:
import ast
from tqdm.auto import tqdm

def trim_completion(completion):
    """Keep the function body: stop at the first top-level (non-indented) line that
    isn't blank and isn't a new def/decorator continuation of the same program."""
    kept = []
    for ln in completion.split("\n"):
        if ln.strip() == "":
            kept.append(ln); continue
        if not ln.startswith((" ", "\t")):          # left the function body
            break
        kept.append(ln)
    return "\n".join(kept).rstrip()

results = []
for ex in tqdm(mbpp, desc=f"MBPP baseline [{MODEL_ID.split('/')[-1]}]"):
    prompt, head = build_prompt(ex)
    if prompt is None:
        continue
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    if ids.shape[1] >= ctx_len - MAX_NEW_TOKENS:
        continue
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    completion = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)  # new tokens only
    body = trim_completion(completion)
    code_str = (head + body).strip()

    if not body.strip():
        s_ok = False   # empty completion counts as invalid
    else:
        try:
            ast.parse(code_str); s_ok = True
        except SyntaxError:
            s_ok = False

    passed, total = run_tests(code_str, ex.get("test_setup_code", "") or "", ex["test_list"])
    results.append({"task_id": ex["task_id"], "syntax_ok": s_ok, "empty": not body.strip(),
                    "passed": passed, "total": total,
                    "full_pass": passed == total and total > 0})

print(f"Done: {len(results)} problems evaluated")

MBPP baseline [gpt2-xl]:   0%|          | 0/500 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1098 > 1024). Running this sequence through the model will result in indexing errors


Done: 499 problems evaluated


## Report + save

In [13]:
import json, math

def wilson_ci(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p = k/n; denom = 1 + z*z/n
    c = (p + z*z/(2*n))/denom
    h = (z/denom)*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (max(0.0, c-h), min(1.0, c+h))

n = len(results)
k_pass   = sum(r["full_pass"] for r in results)
k_syntax = sum(r["syntax_ok"] for r in results)
n_empty  = sum(r["empty"] for r in results)
pass_at_1 = k_pass/n
syntax_validity = k_syntax/n
avg_test_pass = sum(r["passed"]/r["total"] for r in results if r["total"] > 0) / n
ci_pass, ci_syntax = wilson_ci(k_pass, n), wilson_ci(k_syntax, n)

print(f"--- {MODEL_ID} : MBPP baseline (completion-style) ---")
print(f"n = {n} | empty completions = {n_empty}")
print(f"pass@1:          {pass_at_1:.1%}  (95% CI {ci_pass[0]:.1%} – {ci_pass[1]:.1%})")
print(f"avg test pass:   {avg_test_pass:.1%}")
print(f"syntax validity: {syntax_validity:.1%}  (95% CI {ci_syntax[0]:.1%} – {ci_syntax[1]:.1%})")

with open(OUT_FILE, "w") as f:
    json.dump({
        "model_id": MODEL_ID, "params_millions": round(n_params/1e6, 1),
        "split": "test", "num_problems": n, "empty_completions": n_empty,
        "pass_at_1": pass_at_1, "pass_at_1_ci95": ci_pass,
        "avg_test_pass_rate": avg_test_pass,
        "syntax_validity": syntax_validity, "syntax_validity_ci95": ci_syntax,
        "protocol": {
            "decoding": "greedy (do_sample=False)", "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS,
            "decode": "new-tokens-only, skip_special_tokens=True, body trimmed at first top-level line",
            "prompt": "completion-style: comment(task) + comment(first assert) + reference def signature; NO instruction template",
            "syntax": "ast.parse on signature + trimmed body; empty completion counts as invalid",
            "execution": f"forked child (stdout silenced), {TIMEOUT_S}s hard-kill, test_setup_code + full test_list",
        },
        "per_problem": results,
    }, f, indent=2)
print(f"Saved: {OUT_FILE}")

--- openai-community/gpt2-xl : MBPP baseline (completion-style) ---
n = 499 | empty completions = 499
pass@1:          0.0%  (95% CI 0.0% – 0.8%)
avg test pass:   0.0%
syntax validity: 0.0%  (95% CI 0.0% – 0.8%)
Saved: mbpp_baseline_gpt2-xl.json


## Notes

- **Run order suggestion (same-size range first):** `gpt2` → `codeparrot/codeparrot-small` →
  `HuggingFaceTB/SmolLM2-135M` → `EleutherAI/pythia-160m`, then the big ones
  (`openai-community/gpt2-xl`, TinyLlama base). Small models ~10–15 min each; XL/TinyLlama 30–45 min
  (base models rarely emit EOS, so most generations run the full budget).
- The completion prompt + body-trim replaces PocketCoder's fence cut — the footnote for the paper:
  "baseline (completion) models receive the task and one assertion as comments plus the reference
  signature; PocketCoder SFT variants receive the same information in their instruction template;
  decoding, execution, and metrics are identical."
- Expected: codeparrot-small and GPT-2 small near 0% pass@1 (syntax will differ — that's the
  informative column); SmolLM2/Pythia low; TinyLlama possibly high-single-digits.
- Each JSON records `params_millions` — the params column of the baseline table assembles itself
  from the six files.